In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from PIL import Image

2026-03-30 19:49:03.249616: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-30 19:49:03.305674: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-30 19:49:05.121929: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
PROJECT_ROOT = ".."
CSV_PATH = "../data/data_master.csv"
TRAIN_SOURCES = {"spotlight-sphere-data", "two_object", "two_object_training"}

IMG_SIZE = (128, 128)
BATCH_SIZE =32
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
df = pd.read_csv(CSV_PATH, low_memory=False)
df = df[df["dataset_source"].astype(str).isin(TRAIN_SOURCES)].copy()

label_cols = ["spot_color_r", "spot_color_g", "spot_color_b", "spot_energy"]
df = df[df[label_cols].notna().all(axis=1)].copy()

def resolve_existing_image(row: pd.Series) -> str | None:
    for key in ["resolved_image_relpath", "image_relpath"]:
        raw = row.get(key)
        if pd.isna(raw):
            continue

        p = str(raw)
        candidates = [
            os.path.join(PROJECT_ROOT, p),
            os.path.join(PROJECT_ROOT, "data", p),
            os.path.join(PROJECT_ROOT, "data", "spotlight-sphere-data", p),
            os.path.join(PROJECT_ROOT, "data", "two_object", p),
        ]
        for candidate in candidates:
            if os.path.exists(candidate):
                return candidate
    return None

df["image_path"] = df.apply(resolve_existing_image, axis=1)
df = df[df["image_path"].notna()].reset_index(drop=True)
if df.empty:
    raise ValueError("No labeled images found for the requested training sources.")

print("Rows with existing images:", len(df))
print(df["dataset_source"].value_counts())

Rows with existing images: 2350
dataset_source
two_object               1350
spotlight-sphere-data    1000
Name: count, dtype: int64


In [4]:
def load_and_preprocess_image(path: str) -> np.ndarray:
    with Image.open(path) as img:
        img = img.convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
        return np.asarray(img, dtype=np.float32) / 255.0

X_images = np.stack([load_and_preprocess_image(p) for p in df["image_path"]], axis=0)
print("Image tensor shape:", X_images.shape)

Image tensor shape: (2350, 128, 128, 3)


In [5]:
color_cols = ["spot_color_r", "spot_color_g", "spot_color_b"]
energy_col = "spot_energy"
y_color_raw = df[color_cols].to_numpy(dtype=np.float32)
y_energy_log_raw = np.log(df[energy_col].to_numpy(dtype=np.float32)).reshape(-1, 1)
idx = np.arange(len(df))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42)
idx_train, idx_val = train_test_split(idx_train, test_size=0.2, random_state=42)
X_train, X_val, X_test = X_images[idx_train], X_images[idx_val], X_images[idx_test]
y_color_train_raw, y_color_val_raw, y_color_test_raw = y_color_raw[idx_train], y_color_raw[idx_val], y_color_raw[idx_test]
y_energy_train_raw, y_energy_val_raw, y_energy_test_raw = y_energy_log_raw[idx_train], y_energy_log_raw[idx_val], y_energy_log_raw[idx_test]

color_mean = y_color_train_raw.mean(axis=0, keepdims=True)
color_std = y_color_train_raw.std(axis=0, keepdims=True)
color_std[color_std < 1e-8] = 1.0
energy_mean = y_energy_train_raw.mean(axis=0, keepdims=True)
energy_std = y_energy_train_raw.std(axis=0, keepdims=True)
energy_std[energy_std < 1e-8] = 1.0
y_color_train = (y_color_train_raw - color_mean) / color_std
y_color_val = (y_color_val_raw - color_mean) / color_std
y_color_test = (y_color_test_raw - color_mean) / color_std
y_energy_train = (y_energy_train_raw - energy_mean) / energy_std
y_energy_val = (y_energy_val_raw - energy_mean) / energy_std
y_energy_test = (y_energy_test_raw - energy_mean) / energy_std
print("Train/Val/Test:", len(idx_train), len(idx_val), len(idx_test))
print("Training rows by dataset source:")
print(df["dataset_source"].value_counts())


Train/Val/Test: 1504 376 470
Training rows by dataset source:
dataset_source
two_object               1350
spotlight-sphere-data    1000
Name: count, dtype: int64


In [6]:
img_input = keras.Input(shape=(*IMG_SIZE, 3), name="image")
x = layers.Conv2D(32, 3, activation="relu", padding="same")(img_input)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

color_out = layers.Dense(3, name="color_head")(x)
energy_out = layers.Dense(1, name="energy_head")(x)

model = keras.Model(inputs=img_input, outputs=[color_out, energy_out])
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={"color_head": "mse", "energy_head": "mse"},
    loss_weights={"color_head": 1.0, "energy_head": 0.7},
    metrics={"color_head": ["mae"], "energy_head": ["mae"]},
)
model.summary()

callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True)]

history = model.fit(
    X_train,
    {"color_head": y_color_train, "energy_head": y_energy_train},
    validation_data=(X_val, {"color_head": y_color_val, "energy_head": y_energy_val}),
    epochs=100,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

E0000 00:00:1774918161.492679  279003 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1774918161.501521  279003 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 128, 128,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        896 │ image[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ color_head (Dense)  │ (None, 3)         │        387 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ energy_head (Dense) │ (None, 1)         │        129 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 110,276 (430.77 KB)

 Trainable params: 110,276 (430.77 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 12s 229ms/step - color_head_loss: 0.8856 - color_head_mae: 0.7971 - energy_head_loss: 0.9933 - energy_head_mae: 0.7762 - loss: 1.5809 - val_color_head_loss: 0.6155 - val_color_head_mae: 0.6744 - val_energy_head_loss: 0.9928 - val_energy_head_mae: 0.7862 - val_loss: 1.3011
Epoch 2/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 220ms/step - color_head_loss: 0.4557 - color_head_mae: 0.5584 - energy_head_loss: 0.9912 - energy_head_mae: 0.7801 - loss: 1.1495 - val_color_head_loss: 0.2835 - val_color_head_mae: 0.4324 - val_energy_head_loss: 0.9608 - val_energy_head_mae: 0.7423 - val_loss: 0.9449
Epoch 3/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 220ms/step - color_head_loss: 0.3411 - color_head_mae: 0.4759 - energy_head_loss: 0.9265 - energy_head_mae: 0.7439 - loss: 0.9897 - val_color_head_loss: 0.2686 - val_color_head_mae: 0.4118 - val_energy_head_loss: 0.8987 - val_energy_head_mae: 0.7234 - val_loss: 0.8874
Epoch 4/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 10s 220ms/step - color_he

In [7]:
metrics = model.evaluate(
    X_test,
    {"color_head": y_color_test, "energy_head": y_energy_test},
    verbose=0,
)
print("Raw Keras test metrics:", metrics)

pred_color_z, pred_energy_z = model.predict(X_test, verbose=0)

# invert standardization
pred_color = pred_color_z * color_std + color_mean
true_color = y_color_test_raw

pred_energy_log = pred_energy_z * energy_std + energy_mean
true_energy_log = y_energy_test_raw

pred_color = np.clip(pred_color, 0.0, 1.0)
color_mae = np.mean(np.abs(pred_color - true_color), axis=0)
energy_log_mae = np.mean(np.abs(pred_energy_log[:, 0] - true_energy_log[:, 0]))

# convert back to original units
pred_energy = np.exp(pred_energy_log[:, 0])
true_energy = np.exp(true_energy_log[:, 0])
energy_mae = np.mean(np.abs(pred_energy - true_energy))
energy_mape = np.mean(np.abs(pred_energy - true_energy) / np.clip(true_energy, 1e-6, None)) * 100.0

print("Color MAE [r, g, b]:", color_mae)
print(f"Energy MAE (log space): {energy_log_mae:.6f}")
print(f"Energy MAE (original units): {energy_mae:.3f}")
print(f"Energy MAPE (%): {energy_mape:.2f}")


Raw Keras test metrics: [0.5920364260673523, 0.19143496453762054, 0.5826961398124695, 0.3385795056819916, 0.5026020407676697]
Color MAE [r, g, b]: [0.08306447 0.08811086 0.09226913]
Energy MAE (log space): 0.368864
Energy MAE (original units): 1558.327
Energy MAPE (%): 51.65


In [8]:
# save model for later testing
model.save("color_power_predictor.keras")

# single image inference + comparison
sample_idx = int(idx_test[0])
sample_path = df.iloc[sample_idx]["image_path"]
sample_relpath = df.iloc[sample_idx]["image_relpath"]

sample_img = np.expand_dims(load_and_preprocess_image(sample_path), axis=0)
pred_color_z, pred_energy_z = model.predict(sample_img, verbose=0)

pred_color = np.clip((pred_color_z * color_std + color_mean)[0], 0.0, 1.0)
pred_energy_log = (pred_energy_z * energy_std + energy_mean)[0, 0]
pred_energy = float(np.exp(pred_energy_log))

true_row = df.iloc[sample_idx]
true_color = true_row[color_cols].to_numpy(dtype=np.float32)
true_energy = float(true_row[energy_col])
true_energy_log = float(np.log(true_energy))

print("Image:", sample_relpath)
print("Pred color [r,g,b]:", pred_color)
print("True color [r,g,b]:", true_color)
print(f"Pred energy (log): {pred_energy_log:.6f}")
print(f"True energy (log): {true_energy_log:.6f}")
print("Pred energy:", pred_energy)
print("True energy:", true_energy)


Image: two_object/render_0395.png
Pred color [r,g,b]: [0.71521866 0.6085544  0.8139343 ]
True color [r,g,b]: [0.75858635 0.6489124  0.9785603 ]
Pred energy (log): 8.724780
True energy (log): 9.099371
Pred energy: 6153.5234375
True energy: 8949.6572265625
